In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
print("libraries loaded successfully")

libraries loaded successfully


In [3]:
import pandas as pd
df=pd.read_excel("email_spam_dataset.xlsx")


In [4]:
df.head()

,email_id,sender_email,subject,body_preview,date_received,word_count,num_links,has_attachment,num_exclamations,num_uppercase_words,contains_html,is_reply,sender_ip,label,label_text
0,EMAIL_00001,noreply@github.com,Re: Project update,Your monthly bank statement for February is no...,2024-10-17 10:23:08,46,0,0,0,2,0,1,192.168.0.8,0,ham
1,EMAIL_00002,deals@topdiscounts99.net,Click here to claim your reward,"INVESTMENT OPPORTUNITY: Turn $500 into $50,000...",2024-03-08 03:42:04,58,2,0,10,9,1,0,101.81.216.88,1,spam
2,EMAIL_00003,pills@cheapmeds.xyz,You have an unclaimed inheritance,SECRET TRADING STRATEGY: Wall Street insiders ...,2024-02-15 22:03:09,75,2,0,8,4,1,0,111.235.63.250,1,spam
3,EMAIL_00004,noreply@github.com,Reminder: Doctor appointment Friday,"Hi, I'm writing to follow up on the job applic...",2023-12-21 00:39:48,37,2,1,0,2,0,1,192.168.4.21,0,ham
4,EMAIL_00005,noreply@github.com,Happy Birthday!,Following up on our meeting yesterday. I've su...,2024-02-19 17:39:16,110,1,1,0,1,0,1,192.168.10.166,0,ham


In [5]:
df.shape

(10000, 15)

In [7]:
df.columns

Index(['email_id', 'sender_email', 'subject', 'body_preview', 'date_received',
       'word_count', 'num_links', 'has_attachment', 'num_exclamations',
       'num_uppercase_words', 'contains_html', 'is_reply', 'sender_ip',
       'label', 'label_text'],
      dtype='str')

In [8]:
df=df.drop("label_text",axis=1)

In [9]:
df.head()

,email_id,sender_email,subject,body_preview,date_received,word_count,num_links,has_attachment,num_exclamations,num_uppercase_words,contains_html,is_reply,sender_ip,label
0,EMAIL_00001,noreply@github.com,Re: Project update,Your monthly bank statement for February is no...,2024-10-17 10:23:08,46,0,0,0,2,0,1,192.168.0.8,0
1,EMAIL_00002,deals@topdiscounts99.net,Click here to claim your reward,"INVESTMENT OPPORTUNITY: Turn $500 into $50,000...",2024-03-08 03:42:04,58,2,0,10,9,1,0,101.81.216.88,1
2,EMAIL_00003,pills@cheapmeds.xyz,You have an unclaimed inheritance,SECRET TRADING STRATEGY: Wall Street insiders ...,2024-02-15 22:03:09,75,2,0,8,4,1,0,111.235.63.250,1
3,EMAIL_00004,noreply@github.com,Reminder: Doctor appointment Friday,"Hi, I'm writing to follow up on the job applic...",2023-12-21 00:39:48,37,2,1,0,2,0,1,192.168.4.21,0
4,EMAIL_00005,noreply@github.com,Happy Birthday!,Following up on our meeting yesterday. I've su...,2024-02-19 17:39:16,110,1,1,0,1,0,1,192.168.10.166,0


In [10]:
df

,email_id,sender_email,subject,body_preview,date_received,word_count,num_links,has_attachment,num_exclamations,num_uppercase_words,contains_html,is_reply,sender_ip,label
0,EMAIL_00001,noreply@github.com,Re: Project update,Your monthly bank statement for February is no...,2024-10-17 10:23:08,46,0,0,0,2,0,1,192.168.0.8,0
1,EMAIL_00002,deals@topdiscounts99.net,Click here to claim your reward,"INVESTMENT OPPORTUNITY: Turn $500 into $50,000...",2024-03-08 03:42:04,58,2,0,10,9,1,0,101.81.216.88,1
2,EMAIL_00003,pills@cheapmeds.xyz,You have an unclaimed inheritance,SECRET TRADING STRATEGY: Wall Street insiders ...,2024-02-15 22:03:09,75,2,0,8,4,1,0,111.235.63.250,1
3,EMAIL_00004,noreply@github.com,Reminder: Doctor appointment Friday,"Hi, I'm writing to follow up on the job applic...",2023-12-21 00:39:48,37,2,1,0,2,0,1,192.168.4.21,0
4,EMAIL_00005,noreply@github.com,Happy Birthday!,Following up on our meeting yesterday. I've su...,2024-02-19 17:39:16,110,1,1,0,1,0,1,192.168.10.166,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,EMAIL_09996,winner@lucky-prizes.net,Get rich quick - SECRET revealed,"Dear Friend, I am a Nigerian prince with $45 m...",2024-07-18 17:01:01,105,3,0,3,6,1,0,135.211.184.200,1
9996,EMAIL_09997,lottery@intl-winner.com,Re: Your loan application approved,Congratulations! You have been selected as tod...,2023-04-17 21:35:22,106,5,0,9,6,1,0,198.231.104.11,1
9997,EMAIL_09998,it-support@company.com,Reminder: Doctor appointment Friday,Your order #ORD-8842 has been shipped and is e...,2025-02-05 21:22:54,183,0,0,0,2,0,1,192.168.5.82,0
9998,EMAIL_09999,notice@account-alert.info,Buy Viagra online cheap,"Dear Friend, I am a Nigerian prince with $45 m...",2024-05-04 12:30:48,101,4,1,10,6,1,0,188.1.200.56,1


In [11]:
df.shape

(10000, 14)

In [13]:
df['label'].unique()

array([0, 1])

In [14]:
df.isnull().sum()

email_id               0
sender_email           0
subject                0
body_preview           0
date_received          0
word_count             0
num_links              0
has_attachment         0
num_exclamations       0
num_uppercase_words    0
contains_html          0
is_reply               0
sender_ip              0
label                  0
dtype: int64

In [15]:
df.dropna(inplace=True)

In [19]:
#for cleaning body preview
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['body_preview'] = df['body_preview'].apply(clean_text)

In [20]:
#data split
X = df['body_preview']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

In [21]:
#check sizes
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (8000,)
Test size: (2000,)


In [22]:
#data vetorization
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [24]:
#for training data or modal
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("Classification Report:")
print(classification_report(y_test, y_test_pred))

Train Accuracy: 1.0
Test Accuracy: 1.0
Confusion Matrix:
[[1189    0]
 [   0  811]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1189
           1       1.00      1.00      1.00       811

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [25]:
#evaluate
#confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[1189    0]
 [   0  811]]


In [26]:
#classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1189
           1       1.00      1.00      1.00       811

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [27]:
#check Overloading
print("Train Accuracy:", model.score(X_train, y_train))
print("Test Accuracy:", model.score(X_test, y_test))

Train Accuracy: 1.0
Test Accuracy: 1.0


In [29]:
#for saving
import pickle

pickle.dump(model, open("spam_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))